# 네이버지도 블로그 리뷰 수집

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

: 

In [ ]:
options = Options()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

place_name = '신창국밥 본점'
# 네이버 지도 검색 페이지
base_url = f'https://map.naver.com/p/search/{place_name}'
driver.get(base_url)
time.sleep(2)


# driver.switch_to.frame('searchIframe')
# time.sleep(1)
# driver.find_element(By.XPATH, '//*[@id="_pcmap_list_scroll_container"]/ul/li[1]/div[1]/div[2]/div[1]/a/span[1]').click()
# time.sleep(1)
# driver.switch_to.default_content()
# time.sleep(1)
driver.switch_to.frame('entryIframe')
# 댓글 버튼
# driver.find_element(By.XPATH, '//*[@id="app-root"]/div/div/div/div[4]/div/div/div/div/a[3]/span').click()
driver.find_element(By.XPATH, '//a[contains(text(), "리뷰")]').click()
time.sleep(1)
# 블로그 글
driver.find_element(By.XPATH, '//*[@id="_subtab_view"]/div/a[2]').click()
time.sleep(1)

# 최신순
try:
    driver.find_element(By.XPATH, '//*[@id="app-root"]/div/div/div/div[6]/div[3]/div/div[1]/div/div/div[1]/span[2]').click()
except:
    driver.find_element(By.XPATH, '//*[@id="app-root"]/div/div/div/div[6]/div[3]/div/div[1]/div/div/div[1]/span[2]').click()


for _ in range(10):
    # 맨 아래 스크롤
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)
    try:
        driver.find_element(By.XPATH, '//*[@id="app-root"]/div/div/div/div[6]/div[3]/div/div[2]/div/a/span').click()
    except:
        print("더보기 버튼이 없습니다.")
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        break

html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

df = pd.DataFrame(columns=['닉네임', '제목', '작성일', '리뷰'])
for content in soup.find_all('li', class_ = 'EblIP'):
    nickname = content.find('span', class_='pui__NMi-Dp').text
    title = content.find('div', class_='pui__dGLDWy').text
    date = content.find('time').text
    review = content.find('div', class_ = 'pui__vn15t2').find('span').text
    df.loc[len(df)] = [nickname, title, date, review]
df.to_csv(f'./data/reviews/misc/{place_name}_블로그리뷰.csv')

## 함수 + 반복문

In [3]:
def collect_blog_reviews(place_name):
    options = Options()
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    # 네이버 지도 검색 페이지
    base_url = f'https://map.naver.com/p/search/{place_name}'
    driver.get(base_url)
    time.sleep(2)

    # driver.switch_to.frame('searchIframe')
    # time.sleep(1)
    # driver.find_element(By.XPATH, '//*[@id="_pcmap_list_scroll_container"]/ul/li[1]/div[1]/div[2]/div[1]/a/span[1]').click()
    # time.sleep(1)
    # driver.switch_to.default_content()
    # time.sleep(1)
    driver.switch_to.frame('entryIframe')
    # 댓글 버튼
    #driver.find_element(By.XPATH, '//*[@id="app-root"]/div/div/div/div[4]/div/div/div/div/a[3]/span').click()
    driver.find_element(By.XPATH, '//a[contains(text(), "리뷰")]').click()
    time.sleep(1)
    # 블로그 글
    driver.find_element(By.XPATH, '//*[@id="_subtab_view"]/div/a[2]').click()
    time.sleep(2)

    # 최신순
    try:
        driver.find_element(By.XPATH, '//*[@id="app-root"]/div/div/div/div[6]/div[3]/div/div[1]/div/div/div[1]/span[2]').click()
    except:
        driver.find_element(By.XPATH, '//*[@id="app-root"]/div/div/div/div[6]/div[3]/div/div[1]/div/div/div[1]/span[2]').click()

    for _ in range(20):
        # 맨 아래 스크롤
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)
        try:
            driver.find_element(By.XPATH, '//*[@id="app-root"]/div/div/div/div[6]/div[3]/div/div[2]/div/a/span').click()
        except:
            print("더보기 버튼이 없습니다.")
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            break

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    df = pd.DataFrame(columns=['닉네임', '제목', '작성일', '리뷰'])
    for content in soup.find_all('li', class_ = 'EblIP'):
        nickname = content.find('span', class_='pui__NMi-Dp').text
        title = content.find('div', class_='pui__dGLDWy').text
        date = content.find('time').text
        review = content.find('div', class_ = 'pui__vn15t2').find('span').text
        df.loc[len(df)] = [nickname, title, date, review]
    time.sleep(1)
    driver.quit()
    return df

### 수집대상 불러오기

In [7]:
famous_df = pd.read_csv('./data/processed/유명가게.csv')
famous_df.drop(index=[14, 15, 39], inplace=True)
famous_df.reset_index(inplace=True, drop=True)
famous_df.head()

,식당명,주소
0,합천일류돼지국밥,사상구 광장로 34
1,신창국밥 본점,서구 보수대로 53
2,쌍둥이돼지국밥 본점,남구 유엔평화로 35-1
3,60년 전통 할매국밥,동구 중앙대로533번길 4
4,몽실종가돼지국밥 감천문화마을 본점,"서구 까치고개로 197번길 3, 1층"


In [8]:
famous_df['수집여부'] = np.nan

for i in range(len(famous_df)):
    name = famous_df.loc[i].values[0]
    place_name = famous_df.loc[i].values[0] + ' ' + famous_df.loc[i].values[1]
    print(i, name, '수집시작')
    try:
        df = collect_blog_reviews(place_name)
        df.to_csv(f'./data/reviews/misc/{name}_블로그리뷰.csv', index=False)
        famous_df.loc[i ,'수집여부'] = True
        print('수집성공')
    except:
        try:
            df = collect_blog_reviews(name)
            df.to_csv(f'./data/reviews/misc/{name}_블로그리뷰.csv', index=False)
            famous_df.loc[i ,'수집여부'] = True
            print('수집성공')
        except:
            famous_df.loc[i ,'수집여부'] = False
            print('수집실패')


0 합천일류돼지국밥 수집시작
더보기 버튼이 없습니다.


C:\Users\user\AppData\Local\Temp\ipykernel_25664\981189937.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  famous_df.loc[i ,'수집여부'] = True


수집성공
1 신창국밥 본점 수집시작
더보기 버튼이 없습니다.
수집성공
2 쌍둥이돼지국밥 본점 수집시작
더보기 버튼이 없습니다.
수집성공
3 60년 전통 할매국밥 수집시작
더보기 버튼이 없습니다.
수집성공
4 몽실종가돼지국밥 감천문화마을 본점 수집시작
더보기 버튼이 없습니다.
수집성공
5 영진돼지국밥 본점 수집시작
더보기 버튼이 없습니다.
수집성공
6 합천국밥집 수집시작
더보기 버튼이 없습니다.
수집성공
7 정짓간 돼지국밥 & 막국수 수집시작
더보기 버튼이 없습니다.
수집성공
8 우리돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
9 수복돼지국밥 수집시작
수집실패
10 부광돼지국밥전문점 수집시작
더보기 버튼이 없습니다.
수집성공
11 두번째 늘해랑 수집시작
더보기 버튼이 없습니다.
수집성공
12 재기돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
13 자매국밥 수집시작
더보기 버튼이 없습니다.
수집성공
14 송정3대국밥 수집시작
더보기 버튼이 없습니다.
수집성공
15 원산순대돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
16 오가네국밥 수집시작
더보기 버튼이 없습니다.
수집성공
17 바보국밥1호점 수집시작
더보기 버튼이 없습니다.
수집성공
18 세화돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
19 수라국밥 수집시작
더보기 버튼이 없습니다.
수집성공
20 영진돼지국밥 수집시작
수집실패
21 마루돼지국밥 수집시작
수집실패
22 장수돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
23 더도이종가집돼지국밥 수집시작
수집실패
24 합천돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
25 아제돼지국밥 수집시작
수집실패
26 경주박가국밥 수집시작
더보기 버튼이 없습니다.
수집성공
27 굴마을굴국밥 수집시작
수집실패
28 병천순대국밥 수집시작
더보기 버튼이 없습니다.
수집성공
29 또랑돼지국밥 수집시작
수집실패
30 밀양삼대국밥 수집시작
더보기 버튼이 없습니다.
수집성공
31 시장국밥집 수집시작
수집실패
32 가야포차선지국밥 수집시작
수집실패
33 

In [14]:
famous_df.to_csv('./data/processed/유명가게_수집현황.csv', index=False)

In [6]:
total_df = pd.read_csv('./data/processed/전체가게.csv', index_col= 0)
total_df

,식당명,주소
0,설봉돼지국밥,부산광역시 금정구 수림로49번길 6 (장전동)
1,또순이돼지국밥,부산광역시 금정구 서부로 20 (서동)
2,목촌돼지국밥 서동점,부산광역시 금정구 서동로 192 (서동)
3,영진돼지국밥,부산광역시 금정구 개좌로 84 (금사동)
4,장수돼지국밥,부산광역시 금정구 부곡로 96 (부곡동)
...,...,...
88,필순할매돼지국밥,"부산광역시 부산진구 새싹로8번길 16, 1층 (부전동)"
89,참잘돼지국밥,"부산광역시 부산진구 서전로 11-1, 신전빌딩 1층 (부전동)"
90,다복 돼지국밥,"부산광역시 부산진구 냉정로 204, 1층 (개금동)"
91,최고집명가돼지국밥,"부산광역시 부산진구 중앙대로 903-1, 1층 (양정동)"


In [7]:
total_df['수집여부'] = np.nan

for i in range(len(total_df)):
    name = total_df.loc[i].values[0]
    place_name = total_df.loc[i].values[0] + ' ' + total_df.loc[i].values[1]
    print(i, name, '수집시작')
    try:
        df = collect_blog_reviews(place_name)
        df.to_csv(f'./data/reviews/all/{name}_블로그리뷰.csv', index=False)
        total_df.loc[i ,'수집여부'] = True
        print('수집성공')
    except:
        try:
            df = collect_blog_reviews(name)
            df.to_csv(f'./data/reviews/all/{name}_블로그리뷰.csv', index=False)
            total_df.loc[i ,'수집여부'] = True
            print('수집성공')
        except:
            total_df.loc[i ,'수집여부'] = False
            print('수집실패')


0 설봉돼지국밥 수집시작
더보기 버튼이 없습니다.


C:\Users\user\AppData\Local\Temp\ipykernel_23804\1741578435.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  total_df.loc[i ,'수집여부'] = True


수집성공
1 또순이돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
2 목촌돼지국밥 서동점 수집시작
더보기 버튼이 없습니다.
수집성공
3 영진돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
4 장수돼지국밥 수집시작
수집실패
5 별미청 돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
6 행복돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
7 장수돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
8 남산돼지국밥 수집시작
수집실패
9 언양돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
10 소문난돼지국밥 수집시작
수집실패
11 마루돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
12 금문돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
13 우리마을돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
14 무진장돼지국밥 수집시작
수집실패
15 수라돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
16 장수원돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
17 화로돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
18 밀양돼지국밥 수집시작
수집실패
19 서동돼지국밥 수집시작
수집실패
20 정원돼지국밥 수집시작
수집실패
21 가야공원돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
22 미광돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
23 합천돼지국밥 수집시작
수집실패
24 왕대박돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
25 현대보쌈돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
26 금정돼지국밥 수집시작
수집실패
27 복돈돼지국밥 수집시작
수집실패
28 고려돼지국밥 수집시작
수집실패
29 한성 가마솥 돼지국밥 수집시작
더보기 버튼이 없습니다.
수집성공
30 장전돼지국밥 수집시작
수집실패
31 백년 돼지국밥 수집시작
수집실패
32 한뚝배기 돼지국밥 수집시작
수집실패
33 밀양돼지국밥&감자탕 수집시작
수집실패
34 구월산 돼지국밥 수집시작
수집실패
35 청화백 돼지국밥 수집시작
수집실패
36 서면포항돼지국밥 수집시작
더

In [8]:
total_df.to_csv('./data/processed/전체가게_수집현황.csv', index=False)

# 수집 실패 처리

In [6]:
import pandas as pd
df_total = pd.read_csv('./data/processed/전체가게_수집현황.csv')
df_famous = pd.read_csv('./data/processed/유명가게_수집현황.csv')
# df_total = df_total[df_total['수집여부'] == False]
# df_famous = df_famous[df_famous['수집여부'] == False]
df_famous.head()

,식당명,주소,수집여부
0,합천일류돼지국밥,사상구 광장로 34,True
1,신창국밥 본점,서구 보수대로 53,True
2,쌍둥이돼지국밥 본점,남구 유엔평화로 35-1,True
3,60년 전통 할매국밥,동구 중앙대로533번길 4,True
4,몽실종가돼지국밥 감천문화마을 본점,"서구 까치고개로 197번길 3, 1층",True


In [ ]:
for i in range(len(df_famous)):
    if df_famous.loc[i, '수집여부'] == True:
        continue
    else:
        name = df_famous.loc[i].values[0]
        place_name = df_famous.loc[i].values[0] + ' ' + df_famous.loc[i].values[1]
        print(i, name, '수집시작')
        try:
            df = collect_blog_reviews(place_name)
            df.to_csv(f'./data/reviews/famous/{name}_블로그리뷰.csv', index=False)
            df_famous.loc[i ,'수집여부'] = True
            print('수집성공')
        except:
            try:
                df = collect_blog_reviews(name)
                df.to_csv(f'./data/reviews/famous/{name}_블로그리뷰.csv', index=False)
                df_famous.loc[i ,'수집여부'] = True
                print('수집성공')
            except:
                df_famous.loc[i ,'수집여부'] = False
                print('수집실패')

9 수복돼지국밥 수집시작
수집실패
20 영진돼지국밥 수집시작
수집실패
21 마루돼지국밥 수집시작
더보기 버튼이 없습니다.
수집실패
23 더도이종가집돼지국밥 수집시작


: 